# 1. Load data from SuperBase

In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

In [2]:
import os
import yaml
from pathlib import Path

config_path = os.path.join(rag_service_dir, "configs", "config.yaml")
print(f"Đường dẫn file config tuyệt đối: {config_path}")
with open(config_path, "r", encoding="utf-8") as f:
    yaml_data = yaml.safe_load(f)

print(yaml_data)

class AppConfig:
    def __init__(self, raw_yaml):
        # 1. Tham số Chunking
        chunking_sec = raw_yaml.get("chunking", {})
        self.chunk_size = chunking_sec.get("chunk_size", 1000)
        self.chunk_overlap = chunking_sec.get("chunk_overlap", 200)
        
        # 2. Tham số Retrieval
        retrieval_sec = raw_yaml.get("retrieval", {})
        self.k = retrieval_sec.get("k", 5)
        self.graph_max_hops = retrieval_sec.get("graph_max_hops", 2)
        self.rrf_k = retrieval_sec.get("rrf_k", 60)
        
        # 3. Tham số Generation
        generation_sec = raw_yaml.get("generation", {})
        self.temperature = generation_sec.get("temperature", 0.0)
        self.max_tokens = generation_sec.get("max_tokens", 2048)

# Khởi tạo cấu hình chung
config = AppConfig(yaml_data)

# Bạn có thể gọi bất kỳ thuộc tính nào ở bất kỳ đâu:
print(config.chunk_size)        # 1000
print(config.k)                 # 5
print(config.temperature)       # 0.0


Đường dẫn file config tuyệt đối: D:\create\Agenttic-RAG-for-e-commerce\rag-service\configs\config.yaml
{'chunking': {'chunk_size': 1000, 'chunk_overlap': 200}, 'retrieval': {'k': 5, 'graph_max_hops': 2, 'rrf_k': 60}, 'generation': {'temperature': 0.0, 'max_tokens': 2048}}
1000
5
0.0


In [3]:
import os
from typing import List, Dict, Any
from pathlib import Path
from supabase import create_client, Client
from configs.setting import settings

class SupabaseDataLoader:
    """Loader chịu trách nhiệm kết nối và lấy dữ liệu thô từ Supabase Postgres"""
    
    def __init__(self):
        self.supabase_url = settings.SUPABASE_URL
        self.supabase_key = settings.SUPABASE_SERVICE_ROLE_KEY
        
        if not self.supabase_url or not self.supabase_key:
            raise ValueError("Thiếu cấu hình SUPABASE_URL hoặc SUPABASE_SERVICE_ROLE_KEY trong file .env")
            
        self.client: Client = create_client(self.supabase_url, self.supabase_key)

    def load_products(self) -> List[Dict[str, Any]]:
        """Tải toàn bộ danh sách sản phẩm từ bảng products của Supabase"""
        try:
            print("[Loader] Đang tải dữ liệu sản phẩm từ Supabase...")
            products = []
            limit = 1000
            offset = 0
            while True:
                response = self.client.table("products")\
                    .select("*")\
                    .order("id", desc=True)\
                    .range(offset, offset + limit - 1)\
                    .execute()
                
                batch = response.data
                if not batch:
                    break
                products.extend(batch)
                print(f"  -> Đã tải batch {offset} - {offset + len(batch) - 1} ({len(batch)} sản phẩm)")
                if len(batch) < limit:
                    break
                offset += limit
            
            print(f"[Loader] Đã tải thành công tổng cộng {len(products)} sản phẩm.")
            return products
        except Exception as e:
            print(f"[Loader] Lỗi khi tải sản phẩm từ Supabase: {e}")
            return []

    def load_policies(self) -> List[Dict[str, Any]]:
        """Tải toàn bộ danh sách chính sách từ bảng policies của Supabase"""
        try:
            print("[Loader] Đang tải dữ liệu chính sách từ Supabase...")
            response = self.client.table("policies").select("*").execute()
            db_policies = response.data
            
            policies = []
            for p in db_policies:
                # Định dạng dữ liệu tương thích 100% với loader file local cũ
                # nhưng bổ sung thêm ID, Title, Category gốc từ Database để làm metadata RAG tốt hơn
                policies.append({
                    "id": p.get("id"),
                    "title": p.get("title"),
                    "category": p.get("category"),
                    "source_doc": f"{p.get('category')}_{p.get('title').replace(' ', '_')}.md",
                    "content": p.get("content", ""),
                    "file_path": f"supabase://policies/{p.get('id')}"
                })
            
            print(f"[Loader] Đã tải thành công {len(policies)} chính sách từ database.")
            return policies
        except Exception as e:
            print(f"[Loader] Lỗi khi tải chính sách từ Supabase: {e}")
            return []


In [4]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [5]:
print(len(products))
print(len(policies))

1325
3


In [6]:
for item in policies:
    print(item)

{'id': '0d255481-edd4-44ab-bb06-0f9c6b1a2de5', 'title': 'Chính sách đổi trả 1-đổi-1 sản phẩm công nghệ', 'category': 'return_refund', 'source_doc': 'return_refund_Chính_sách_đổi_trả_1-đổi-1_sản_phẩm_công_nghệ.md', 'content': 'Chính sách đổi trả sản phẩm tại Website TMĐT Điện tử:\n1. Thời gian áp dụng: Trong vòng 30 ngày kể từ ngày nhận hàng thành công.\n2. Điều kiện áp dụng đổi trả:\n- Sản phẩm bị lỗi kỹ thuật (lỗi phần cứng) từ phía nhà sản xuất.\n- Sản phẩm còn nguyên hộp, đầy đủ phụ kiện đi kèm, hóa đơn mua hàng và phiếu bảo hành.\n- Sản phẩm không bị trầy xước, nứt vỡ, ẩm ướt hoặc có dấu hiệu tự ý can thiệp phần cứng.\n3. Hình thức đổi trả:\n- Đổi mới 1-đổi-1 sản phẩm cùng model hoặc nâng cấp lên model cao hơn (bù chênh lệch).\n- Trường hợp hết hàng đổi mới, hỗ trợ hoàn tiền 100% giá trị sản phẩm trên hóa đơn mua hàng.', 'file_path': 'supabase://policies/0d255481-edd4-44ab-bb06-0f9c6b1a2de5'}
{'id': '9c84d9f9-f758-4273-bbba-c6ca133c4556', 'title': 'Chính sách bảo hành chính hãng th

In [7]:
# In sản phẩm đầu tiên
first_product = products[700]
print(first_product)

# Hoặc in đẹp hơn bằng thư viện json có sẵn để dễ nhìn cấu trúc
import json
print(json.dumps(first_product, indent=4, ensure_ascii=False))

{'id': 95959, 'name': 'OPPO Reno13 F 8GB 256GB', 'brand': 'OPPO', 'sku': 'dien-thoai-oppo-reno13f-4g', 'url': 'https://cellphones.com.vn/dien-thoai-oppo-reno13f-4g.html', 'category': 'phone', 'price': 8990000, 'special_price': 8530000, 'final_price': 8530000, 'stock': 0, 'thumbnail': '/d/i/dien-thoai-oppo-reno13f-4g_2_.png', 'image_url': 'https://cdn2.cellphones.com.vn/insecure/rs:fill:358:358/q:90/plain/https://cellphones.com.vn/media/catalog/product/d/i/dien-thoai-oppo-reno13f-4g_2_.png', 'cpu': None, 'ram': '8G', 'storage': '256 GB', 'display_size': '6.67 inches', 'display_resolution': '1080 x 2400 pixels (FullHD+)', 'battery': '5800mAh (Typ)', 'os': 'ColorOS 15, nền tảng Android 15', 'gpu': 'ARM G57 MC2 1.0GHz', 'weight': 'Khoảng 192g (Bao gồm pin)', 'dimensions': '162.2 x 75.05 x 7.76mm', 'included_accessories': 'Điện thoại\nSạc\nCáp USB\nDụng cụ lấy SIM\nHướng dẫn nhanh\nHướng dẫn an toàn\nỐp lưng bảo vệ', 'camera_primary': 'Camera chính: 50MP, F/1.8\nCamera góc rộng: 8MP, F/2.2\

In [8]:
# In sản phẩm đầu tiên
first_product = products[490]
print(first_product)

# Hoặc in đẹp hơn bằng thư viện json có sẵn để dễ nhìn cấu trúc
import json
print(json.dumps(first_product, indent=4, ensure_ascii=False))


{'id': 109879, 'name': 'Laptop Acer Gaming Nitro ProPanel ANV15-41-R9M1', 'brand': 'Acer', 'sku': 'laptop-acer-gaming-nitro-v-15-propanel-anv15-41-r9m1', 'url': 'https://cellphones.com.vn/laptop-acer-gaming-nitro-v-15-propanel-anv15-41-r9m1.html', 'category': 'laptop', 'price': 30990000, 'special_price': 24990000, 'final_price': 24990000, 'stock': 29, 'thumbnail': '/g/r/group_973_1.png', 'image_url': 'https://cdn2.cellphones.com.vn/insecure/rs:fill:358:358/q:90/plain/https://cellphones.com.vn/media/catalog/product/g/r/group_973_1.png', 'cpu': 'AMD Ryzen 5 7535HS (6 lõi / 12 luồng, Base Clock: 3.3 GHz, Max. Boost Clock: Up to 4.55 GHz, 3MB L2 Cache; 16MB L3 Cache)', 'ram': '16GB', 'storage': '512GB', 'display_size': '15.6 inches', 'display_resolution': '1920 x 1080 pixels (FullHD)', 'battery': 'Pin Li-ion 4 cell 57Whr\nBộ đổi nguồn AC 135W', 'os': 'Windows 11 Home Single Language', 'gpu': 'RTX 3050', 'weight': '2.1 kg', 'dimensions': '362.3 x 239.89 x 23.5 mm (Dài x Rộng x Dày)', 'inclu

In [9]:
# In sản phẩm đầu tiên
first_product = products[100]
print(first_product)

# Hoặc in đẹp hơn bằng thư viện json có sẵn để dễ nhìn cấu trúc
import json
print(json.dumps(first_product, indent=4, ensure_ascii=False))


{'id': 130727, 'name': 'Laptop Lenovo IdeaPad 5 2-in-1 14IPH11 83UG0027VN', 'brand': 'Lenovo', 'sku': 'laptop-lenovo-ideapad-5-2-in-1-14iph11-83ug0027vn', 'url': 'https://cellphones.com.vn/laptop-lenovo-ideapad-5-2-in-1-14iph11-83ug0027vn.html', 'category': 'laptop', 'price': 37990000, 'special_price': 35590000, 'final_price': 35590000, 'stock': 9, 'thumbnail': '/t/e/text_ng_n_16_36.png', 'image_url': 'https://cdn2.cellphones.com.vn/insecure/rs:fill:358:358/q:90/plain/https://cellphones.com.vn/media/catalog/product/t/e/text_ng_n_16_36.png', 'cpu': 'Intel Core Ultra 7 355, 8 lõi (4P + 4LPE) / 8 luồng, Max Turbo up to 4.7GHz, 12MB Intel Smart Cache', 'ram': '16GB', 'storage': '512GB', 'display_size': '14 inches', 'display_resolution': '1920 x 1200 pixels (WUXGA)', 'battery': '60Wh, 65W USB-C (3-pin)', 'os': 'Windows 11 Home Single Language, English', 'gpu': None, 'weight': '1.54 kg', 'dimensions': '311.6 x 224.9 x 17.4 mm (Dài x Rộng x Dày)', 'included_accessories': 'Bộ nguồn, máy, bút, 

In [10]:
import pandas as pd

df = pd.DataFrame(products)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1325 entries, 0 to 1324
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    1325 non-null   int64  
 1   name                  1325 non-null   str    
 2   brand                 1322 non-null   str    
 3   sku                   1325 non-null   str    
 4   url                   1325 non-null   str    
 5   category              1325 non-null   str    
 6   price                 1325 non-null   int64  
 7   special_price         1325 non-null   int64  
 8   final_price           1325 non-null   int64  
 9   stock                 1325 non-null   int64  
 10  thumbnail             1325 non-null   str    
 11  image_url             1325 non-null   str    
 12  cpu                   1243 non-null   str    
 13  ram                   1254 non-null   str    
 14  storage               1293 non-null   str    
 15  display_size          1317 non-n

# 2. Preprocess, Chunking and Embedding

## 2.1. Preprocess JSON
- Với thông tin sản phẩm, tiền xử lý các trường JSON và gộp chung vào 1 đoạn chunking
- Với thông tin chính sách, bảo hành, chunking bình thường

In [11]:
import re
from typing import Tuple, Optional

def extract_dimensions(dim_str: Optional[str]) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    """
    Trích xuất 3 số thực (Dài, Rộng, Dày) từ chuỗi kích thước bất kỳ.
    Trả về tuple: (Dài, Rộng, Dày) dưới dạng số thực (float).
    Nếu không tìm đủ 3 số hoặc chuỗi trống, trả về (None, None, None).
    """
    if not dim_str or not isinstance(dim_str, str):
        return None, None, None
    
    # Tìm tất cả các số thực hoặc số nguyên có trong chuỗi
    numbers = re.findall(r"\d+(?:\.\d+)?", dim_str)
    
    # Kiểm tra nếu tìm thấy ít nhất 3 con số
    if len(numbers) >= 3:
        try:
            length = float(numbers[0])     
            width = float(numbers[1])      
            thickness = float(numbers[2])  
            return length, width, thickness
        except ValueError:
            pass
            
    return None, None, None

In [12]:
import json
from typing import Dict, Any, List

def format_product_to_text(prod: Dict[str, Any]) -> str:
    """
    Hàm phân tích object JSON sản phẩm và ghép nối thành một 
    đoạn văn bản thống nhất (Unified Text Document).
    """
    # 1. Lấy và định dạng các thông tin cơ bản cấp 1
    name = prod.get("name", "Không rõ tên")
    brand = prod.get("brand", "Không rõ thương hiệu")
    category = prod.get("category", "Sản phẩm")
    
    # Định dạng hiển thị giá tiền
    price_val = prod.get("price")  or 0
    final_price_val = prod.get("final_price") or prod.get("special_price") or price_val
    discount_val = prod.get("discount") or 0
    
    price_text = f"{final_price_val:,.0f} VNĐ" if final_price_val > 0 else "Liên hệ"
    original_price_text = f"{price_val:,.0f} VNĐ" if price_val > 0 else "Liên hệ"
    discount_text = f"{discount_val}%" if discount_val > 0 else "Không giảm giá"
    
    # Tồn kho
    stock = prod.get("stock", 0)
    stock_text = f"Còn hàng ({stock} sản phẩm)" if stock > 0 else "Hết hàng"
    
    # 2. Phân tích và làm sạch phần thông số kỹ thuật (specs)

    # Lấy thông số từ trường specs phụ, nếu specs rỗng thì lấy từ specs cấp 1
    specs_dict = prod.get("specs", {}) or {}
    
    # Vi xử lý (CPU, Chipset)
    cpu = prod.get("cpu")
    if not cpu and "Chipset" in specs_dict:
        cpu = specs_dict.get("Chipset")

    # GPU, Card đồ họa
    gpu = prod.get("gpu") or specs_dict.get("vga") or specs_dict.get("laptop_vga_filter")
    # RAM 
    ram = prod.get("ram") or specs_dict.get("RAM")

    # Bộ nhớ lưu trữ (cho laptop/điện thoại)
    storage = specs_dict.get("o_cung_laptop") or prod.get("storage") or specs_dict.get("Luu tru")

    # Thôg tin màn hình
    display_size = prod.get("display_size") or specs_dict.get("Man hinh")
    display_resolution = prod.get("display_resolution") or specs_dict.get("Do phan giai")
    display_type = specs_dict.get("display_type") or prod.get("display_type") or None
    refresh_rate = (
        prod.get("refresh_rate")
        or specs_dict.get("laptop_tan_so_quet") 
        or specs_dict.get("refresh_rate") 
        or None
    )
    panel_type = (
        specs_dict.get("laptop_tam_nen_man_hinh") 
        or specs_dict.get("Công nghệ màn hình") 
        or prod.get("display_type") 
        or specs_dict.get("display_type") 
        or prod.get("panel_type") 
        or specs_dict.get("panel_type") 
        or None
    )

    # Pin thiết bị
    battery = prod.get("battery") or specs_dict.get("Pin") or specs_dict.get("battery")

    # Hệ điều hành OS
    os = prod.get("os") or specs_dict.get("HDH") or None

    # Cân nặng
    weight = prod.get("weight") or specs_dict.get("product_weight") or None

    # Kích thước thực tế (Dài x Rộng X Dày)
    raw_dim = prod.get("dimensions") or specs_dict.get("Kich thuoc")
    dimensions = extract_dimensions(raw_dim)

    # Camera
    camera_primary = prod.get("camera_primary") or specs_dict.get("Camera sau") or None
    camera_secondary = prod.get("camera_secondary") or specs_dict.get("Camera truoc") or specs_dict.get("laptop_camera_webcam") or None
    camera_video = prod.get("camera_video") 

    # Công nghệ âm thanh (hiện chỉ cho laptop)
    audio = specs_dict.get("laptop_cong_nghe_am_thanh") or specs_dict.get("audio") or None

    # Bluetooth
    bluetooth = specs_dict.get("Bluetooth") or None

    # Phụ kiện kèm theo
    included_accessories = prod.get("included_accessories") or specs_dict.get("included_accessories") or None
    
    # Tình trạng sản phẩm (cho laptop trước)
    product_state = specs_dict.get("product_state") or None

    # Miêu tả về sản phẩm
    key_selling_points = prod.get("key_selling_points") or None

    # Mô tả chi tiết về sản phẩm (tóm tắt)
    description = prod.get("description") or None

    # ---------------- Dành riêng cho laptop
    # Lấy loại đèn nền bàn phím
    keyboard_backlight = specs_dict.get("laptop_loai_den_ban_phim") or None
    # npu (Neural Processing Unit)
    npu = specs_dict.get("npu") or None
    # wifi
    wlan = specs_dict.get("wlan") or prod.get("wlan") or None
    # Cổng kết nối
    ports_slots = specs_dict.get("ports_slots") or None
    # Loại bảo mật
    security = specs_dict.get("laptop_bao_mat") or None
    # Chất liệu máy
    material = specs_dict.get("laptop_chat_lieu") or None
    # Số khe ram
    ram_slots = specs_dict.get("laptop_so_khe_ram") or None
    # Loại ram
    ram_type = specs_dict.get("laptop_loai_ram") or None
    # Khe đọc thẻ nhớ
    card_reader = specs_dict.get("laptop_khe_doc_the_nho") or None
    # Các tính năng đặc biệt
    laptop_special_feature = specs_dict.get("laptop_special_feature") or None
    # AI hỗ trợ
    ai_standard = specs_dict.get("laptop_cong_nghe_ai_filter") or None
    # Lấy tác vụ/nhu cầu khuyên dùng
    usage_overall = specs_dict.get("nhu_cau_su_dung") or None
    usage_detail = specs_dict.get("laptop_filter_tac_vu_su_dung") or None
    if usage_overall and usage_detail:
        recommended_usage = f"{usage_overall} (Tối ưu cho: {usage_detail})"
    elif usage_overall:
        recommended_usage = usage_overall
    else:
        recommended_usage = usage_detail or None

    # === 4. Ghép nối tất cả thành một đoạn văn bản có cấu trúc ===
    specs_lines = []
    
    # Gom các thông số chung của cả laptop và điện thoại
    if cpu: specs_lines.append(f"- Bộ vi xử lý (CPU/Chipset): {cpu}")
    if ram: specs_lines.append(f"- Dung lượng RAM: {ram}")
    if ram_type: specs_lines.append(f"- Chuẩn/Loại RAM: {ram_type}")
    if ram_slots: specs_lines.append(f"- Số khe cắm RAM: {ram_slots}")
    if storage: specs_lines.append(f"- Dung lượng lưu trữ: {storage}")
    if display_size: specs_lines.append(f"- Kích thước màn hình: {display_size}")
    if display_resolution: specs_lines.append(f"- Độ phân giải màn hình: {display_resolution}")
    if display_type: specs_lines.append(f"- Công nghệ màn hình: {display_type}")
    if panel_type: specs_lines.append(f"- Loại tấm nền màn hình: {panel_type}")
    if refresh_rate: specs_lines.append(f"- Tần số quét màn hình: {refresh_rate}")
    if battery: specs_lines.append(f"- Dung lượng Pin: {battery}")
    if gpu: specs_lines.append(f"- Card đồ họa (GPU/VGA): {gpu}")
    if os: specs_lines.append(f"- Hệ điều hành: {os}")
    if weight: specs_lines.append(f"- Trọng lượng: {weight}")
    
    # Định dạng kích thước nếu là tuple từ hàm extract_dimensions
    if dimensions:
        if isinstance(dimensions, (tuple, list)) and len(dimensions) >= 3:
            specs_lines.append(f"- Kích thước thiết bị: {dimensions[0]} x {dimensions[1]} x {dimensions[2]} mm")
        else:
            specs_lines.append(f"- Kích thước thiết bị: {dimensions}")
            
    # Xử lý làm sạch dấu xuống dòng \n của Camera, Phụ kiện, Cổng kết nối để văn bản nhúng liền mạch
    if camera_primary: 
        specs_lines.append(f"- Camera chính (sau): {str(camera_primary).replace('\n', ', ')}")
    if camera_secondary: 
        specs_lines.append(f"- Camera phụ / Webcam: {str(camera_secondary).replace('\n', ', ')}")
    if camera_video: 
        specs_lines.append(f"- Tính năng quay phim: {str(camera_video).replace('\n', ', ')}")
    if audio: 
        specs_lines.append(f"- Công nghệ âm thanh: {str(audio).replace('\n', ', ')}")
    if bluetooth: 
        specs_lines.append(f"- Kết nối Bluetooth: {bluetooth}")
    if wlan: 
        specs_lines.append(f"- Kết nối Wi-Fi: {wlan}")
    if ports_slots: 
        specs_lines.append(f"- Cổng kết nối & khe cắm:\n  {str(ports_slots).replace('\n', '\n  ')}")
    if card_reader: 
        specs_lines.append(f"- Khe đọc thẻ nhớ: {card_reader}")
    if keyboard_backlight: 
        specs_lines.append(f"- Đèn nền bàn phím: {keyboard_backlight}")
    if npu: 
        specs_lines.append(f"- Bộ xử lý AI (NPU): {npu}")
    if security: 
        specs_lines.append(f"- Công nghệ bảo mật: {security}")
    if material: 
        specs_lines.append(f"- Chất liệu vỏ máy: {material}")
    if laptop_special_feature: 
        specs_lines.append(f"- Tính năng đặc biệt khác: {laptop_special_feature}")
    if ai_standard: 
        specs_lines.append(f"- Tiêu chuẩn công nghệ AI: {ai_standard}")
    if recommended_usage: 
        specs_lines.append(f"- Nhu cầu/Tác vụ tối ưu: {recommended_usage}")
    if product_state: 
        specs_lines.append(f"- Tình trạng sản phẩm: {str(product_state).replace('\n', ', ')}")
    if included_accessories: 
        specs_lines.append(f"- Phụ kiện kèm theo trong hộp: {str(included_accessories).replace('\n', ', ')}")

    specs_text = "\n".join(specs_lines)

    # Ghép nối thành các đoạn văn bản lớn
    unified_text_parts = [
        f"Sản phẩm: {name}",
        f"Thương hiệu: {brand} | Danh mục: {category}",
        f"Thông tin giá & Kho hàng:\n- Giá thực tế: {price_text}\n- Giá gốc niêm yết: {original_price_text}\n- Mức giảm giá: {discount_text}\n- Tình trạng: {stock_text}"
    ]
    
    if specs_text:
        unified_text_parts.append(f"Thông số kỹ thuật chi tiết:\n{specs_text}")
        
    if key_selling_points and key_selling_points.strip():
        unified_text_parts.append(f"Đặc điểm nổi bật:\n{key_selling_points.strip()}")
        
    if description and description.strip():
        unified_text_parts.append(f"Mô tả chi tiết sản phẩm:\n{description.strip()}")
        
    # Nối tất cả các phần lại bằng dấu xuống dòng kép
    return "\n\n".join(unified_text_parts)


In [13]:
# Tiến hành ghép nối
sample_chunk_text = format_product_to_text(products[100])

# In kết quả kiểm tra
print(sample_chunk_text)

Sản phẩm: Laptop Lenovo IdeaPad 5 2-in-1 14IPH11 83UG0027VN

Thương hiệu: Lenovo | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 35,590,000 VNĐ
- Giá gốc niêm yết: 37,990,000 VNĐ
- Mức giảm giá: 6.32%
- Tình trạng: Còn hàng (9 sản phẩm)

Thông số kỹ thuật chi tiết:
- Bộ vi xử lý (CPU/Chipset): Intel Core Ultra 7 355, 8 lõi (4P + 4LPE) / 8 luồng, Max Turbo up to 4.7GHz, 12MB Intel Smart Cache
- Dung lượng RAM: 16GB
- Chuẩn/Loại RAM: SODIMM DDR5-5600
- Số khe cắm RAM: 2 khe (2 x 8GB, nâng cấp tối đa 32GB)
- Dung lượng lưu trữ: 512GB SSD M.2 2242 PCIe 4.0x4 NVMe (Hỗ trợ tối đa hai ổ đĩa, 2 ổ SSD M.2, dung lượng lên đến 1TB)
- Kích thước màn hình: 14 inches
- Độ phân giải màn hình: 1920 x 1200 pixels (WUXGA)
- Công nghệ màn hình: Độ sáng 400 nits
Độ phủ màu 45% NTSC
Low Blue Light (TÜV)
Mặt kính (Glass)
- Loại tấm nền màn hình: Tấm nền IPS
- Tần số quét màn hình: 60 Hz
- Dung lượng Pin: 60Wh, 65W USB-C (3-pin)
- Card đồ họa (GPU/VGA): Intel Graphics
- Hệ điều hành: Windows 11 

## 2.2. Test chunking and embedding with NVIDIA: Llama Nemotron Embed VL 1B V2

In [ ]:
import requests
import json
from configs.setting import settings

# Lấy API Key tự động từ cấu hình hệ thống
api_key = settings.OPENROUTER_API_KEY

if not api_key or "your-openrouter" in api_key:
    raise ValueError("Vui lòng điền OPENROUTER_API_KEY thật vào file .env!")

# Tạo payload đơn giản (truyền chuỗi văn bản trực tiếp vào "input")
payload = {
    "model": settings.EMBEDDING_MODEL_NAME,
    "input": sample_chunk_text,
    "encoding_format": "float"
}

response = requests.post(
    url="https://openrouter.ai/api/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    data=json.dumps(payload)
)

result = response.json()
print(json.dumps(result, indent=2))

# Kiểm tra kết quả
if "error" in result:
    print("Lỗi từ OpenRouter API:", result["error"])
else:
    embedding = result["data"][0]["embedding"]
    print(f"Nhúng thành công! Số chiều của Vector (Dimension): {len(embedding)}")
    print("5 giá trị đầu tiên của Vector:", embedding[:5])


{
  "object": "list",
  "data": [
    {
      "object": "embedding",
      "embedding": [
        -0.015899658203125,
        -0.013824462890625,
        0.03515625,
        0.03350830078125,
        0.00787353515625,
        0.045196533203125,
        0.003780364990234375,
        0.0157318115234375,
        0.020538330078125,
        0.0006055831909179688,
        -0.003993988037109375,
        0.0107574462890625,
        -0.00958251953125,
        0.0108795166015625,
        0.02880859375,
        -0.010528564453125,
        0.0261688232421875,
        0.023193359375,
        -0.005146026611328125,
        0.0203399658203125,
        0.00710296630859375,
        -0.01064300537109375,
        0.03497314453125,
        -0.014068603515625,
        0.00965118408203125,
        0.01181793212890625,
        0.036468505859375,
        -0.0085601806640625,
        0.05828857421875,
        -0.00414276123046875,
        0.03350830078125,
        -0.010711669921875,
        0.0106887817382812

# 3. Chunking, embedding and load in ChromaDB

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, Any, List

class ChunkingDocuments:
    def __init__(self, config):
        self.config = config

    def build_splitter(self):
        return RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

    def chunk_policy(self, doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Chia nhỏ 1 sản phẩm/chính sách dạng Dictionary"""
        policy_chunks = []
        
        # 1. Khởi tạo splitter (không truyền self.config vào đây nữa vì đã dùng trong build_splitter)
        splitter = self.build_splitter()
        
        # 2. Ghép tiêu đề và nội dung để giữ ngữ cảnh trước khi chia nhỏ
        doc_text = f"Tài liệu chính sách: {doc['title']}\nNội dung chi tiết:\n{doc['content']}"
        
        # 3. Sử dụng split_text cho chuỗi văn bản thô
        chunks = splitter.split_text(doc_text)

        # 4. Đóng gói danh sách chunk
        for i, chunk_content in enumerate(chunks):
            policy_chunks.append({
                "id": f"policy_{doc['id']}_chunk_{i}",  
                "document": chunk_content,            
                "metadata": {
                    "policy_id": str(doc["id"]),
                    "title": str(doc["title"]),
                    "category": str(doc["category"]),
                    "chunk_index": int(i)
                }
            })
        return policy_chunks

    def chunk_multiple_policies(self, docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Hàm bổ sung: Chia nhỏ hàng loạt chính sách cùng lúc"""
        all_chunks = []
        for doc in docs:
            all_chunks.extend(self.chunk_policy(doc))
        return all_chunks




c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any
from configs.setting import settings

# Khởi tạo kết nối ChromaDB
persist_directory = settings.vector_db_absolute_path
client = chromadb.PersistentClient(path=persist_directory)

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

# =====================================================================
# 2. CÁC HÀM TRỢ GIÚP (LÀM SẠCH LINK ẢNH & LẤY VECTOR NHÚNG)
# =====================================================================

def clean_image_urls(text: str) -> str:
    """Xóa sạch link ảnh tuyệt đối/tương đối để tránh lỗi API OpenRouter nhận nhầm"""
    if not text:
        return ""
    text = re.sub(r'https?://\S+\.(?:png|jpg|jpeg|gif|webp)\S*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'/\S+\.(?:png|jpg|jpeg|gif|webp)', '', text, flags=re.IGNORECASE)
    return text

def get_embedding_from_openrouter(text: str, api_key: str) -> List[float]:
    """Gọi API OpenRouter lấy vector nhúng 2048 chiều"""
    cleaned_text = clean_image_urls(text)
    url = "https://openrouter.ai/api/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "nvidia/llama-nemotron-embed-vl-1b-v2:free",
        "input": cleaned_text,
        "encoding_format": "float"
    }
    
    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=15)
            result = response.json()
            if "data" in result:
                return result["data"][0]["embedding"]
            else:
                print(f"Lỗi API: {result.get('error')}. Đang thử lại lần {attempt + 1}...")
        except Exception as e:
            print(f"Lỗi kết nối: {e}. Đang thử lại lần {attempt + 1}...")
        time.sleep(2)
    raise ValueError("Không thể lấy embedding sau 3 lần thử!")

# Khởi tạo API key
api_key = settings.OPENROUTER_API_KEY
if not api_key or "your-openrouter" in api_key:
    raise ValueError("Vui lòng điền OPENROUTER_API_KEY thật vào file .env!")

start_time = time.time()

# =====================================================================
# 3. QUY TRÌNH NẠP 1: SẢN PHẨM (PRODUCTS) - Nạp theo từng đợt (Batch 50)
# =====================================================================
print(f"\n[1/2] Bắt đầu nạp {len(products)} sản phẩm vào 'products_collection'...")

batch_size = 50
batch_ids = []
batch_embeddings = []
batch_documents = []
batch_metadatas = []

for idx, prod in enumerate(products):
    try:
        # Ghép nối JSON thành văn bản phẳng
        doc_text = format_product_to_text(prod)
        
        # Nhúng vector
        vector = get_embedding_from_openrouter(doc_text, api_key)
        
        # Làm sạch metadata tránh lỗi kiểu dữ liệu của ChromaDB
        metadata = {
            "product_id": str(prod.get("id") or ""),
            "brand": str(prod.get("brand") or "Unknown"),
            "category": str(prod.get("category") or "Unknown"),
            "price": float(prod.get("final_price") or prod.get("price") or 0.0)
        }
        
        batch_ids.append(f"prod_{prod['id']}")
        batch_embeddings.append(vector)
        batch_documents.append(doc_text)
        batch_metadatas.append(metadata)
        
        time.sleep(0.1)
    except Exception as ex:
        print(f"Gặp lỗi tại sản phẩm ID {prod.get('id')}: {ex}")
        continue

    # Tiến hành nạp theo batch
    if len(batch_ids) == batch_size or idx == len(products) - 1:
        if batch_ids:
            try:
                product_col.add(
                    ids=batch_ids,
                    embeddings=batch_embeddings,
                    documents=batch_documents,
                    metadatas=batch_metadatas
                )
                print(f"  -> Đã nạp thành công đợt {idx + 1}/{len(products)} sản phẩm.")
            except Exception as db_err:
                print(f"Lỗi nạp batch sản phẩm: {db_err}")
            
            # Reset batch
            batch_ids.clear()
            batch_embeddings.clear()
            batch_documents.clear()
            batch_metadatas.clear()

# =====================================================================
# 4. QUY TRÌNH NẠP 2: CHÍNH SÁCH (POLICIES) - Cắt chunk và nạp tất cả
# =====================================================================
# Giả sử bạn đã load chính sách từ database vào biến `policies` ở cell trước
if 'policies' in locals() and policies:
    print(f"\n[2/2] Bắt đầu phân tích và nạp chính sách vào 'policies_collection'...")
    
    # Khởi tạo class cắt chunk OOP (đã định nghĩa ở cell trước)
    chunker = ChunkingDocuments(config)
    
    # Tiến hành cắt nhỏ toàn bộ chính sách
    policy_chunks = chunker.chunk_multiple_policies(policies)
    print(f"  -> Đã chia nhỏ chính sách thành {len(policy_chunks)} mảnh nhỏ (chunks).")
    
    p_ids = []
    p_embeddings = []
    p_documents = []
    p_metadatas = []
    
    for c_idx, chunk in enumerate(policy_chunks):
        try:
            # Nhúng vector cho từng mảnh chính sách
            vector = get_embedding_from_openrouter(chunk["document"], api_key)
            
            p_ids.append(chunk["id"])
            p_embeddings.append(vector)
            p_documents.append(chunk["document"])
            p_metadatas.append(chunk["metadata"])
            
            print(f"  -> Đã nhúng xong mảnh {c_idx + 1}/{len(policy_chunks)}.")
            time.sleep(0.1)
        except Exception as ex:
            print(f"Lỗi nhúng mảnh chính sách thứ {c_idx}: {ex}")
            
    # Nạp toàn bộ các mảnh chính sách vào collection riêng biệt
    if p_ids:
        try:
            policy_col.add(
                ids=p_ids,
                embeddings=p_embeddings,
                documents=p_documents,
                metadatas=p_metadatas
            )
            print(f"  -> Đã nạp thành công {len(p_ids)} mảnh chính sách vào VectorDB.")
        except Exception as db_err:
            print(f"Lỗi nạp chính sách vào ChromaDB: {db_err}")
else:
    print("\n[2/2] Bỏ qua nạp chính sách (Biến 'policies' trống hoặc chưa được khởi tạo).")

# =====================================================================
# KẾT THÚC
# =====================================================================
end_time = time.time()
print(f"\n==========================================")
print(f"=== HOÀN THÀNH QUY TRÌNH NẠP VECTOR RAG ===")
print(f"Tổng thời gian thực thi: {round(end_time - start_time, 2)} giây.")
print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")
print(f"==========================================")



[1/2] Bắt đầu nạp 1325 sản phẩm vào 'products_collection'...
  -> Đã nạp thành công đợt 50/1325 sản phẩm.
  -> Đã nạp thành công đợt 100/1325 sản phẩm.
  -> Đã nạp thành công đợt 150/1325 sản phẩm.
  -> Đã nạp thành công đợt 200/1325 sản phẩm.
  -> Đã nạp thành công đợt 250/1325 sản phẩm.
  -> Đã nạp thành công đợt 300/1325 sản phẩm.
  -> Đã nạp thành công đợt 350/1325 sản phẩm.
  -> Đã nạp thành công đợt 400/1325 sản phẩm.
  -> Đã nạp thành công đợt 450/1325 sản phẩm.
  -> Đã nạp thành công đợt 500/1325 sản phẩm.
  -> Đã nạp thành công đợt 550/1325 sản phẩm.
  -> Đã nạp thành công đợt 600/1325 sản phẩm.
  -> Đã nạp thành công đợt 650/1325 sản phẩm.
  -> Đã nạp thành công đợt 700/1325 sản phẩm.
  -> Đã nạp thành công đợt 750/1325 sản phẩm.
  -> Đã nạp thành công đợt 800/1325 sản phẩm.
  -> Đã nạp thành công đợt 850/1325 sản phẩm.
  -> Đã nạp thành công đợt 900/1325 sản phẩm.
  -> Đã nạp thành công đợt 950/1325 sản phẩm.
  -> Đã nạp thành công đợt 1000/1325 sản phẩm.
  -> Đã nạp thành 

In [20]:
import chromadb

# 2. Liệt kê toàn bộ các Collection và số lượng vector bên trong
collections = client.list_collections()
print(f"Tổng số collection hiện có: {len(collections)}\n")

for col in collections:
    # col.count() trả về số lượng vector (chunk) trong collection này
    print(f"Collection: {col.name}")
    print(f"  -> Số lượng vector (chunks): {col.count()}")


Tổng số collection hiện có: 2

Collection: policies_collection
  -> Số lượng vector (chunks): 3
Collection: products_collection
  -> Số lượng vector (chunks): 1325


In [21]:
# Lấy collection cụ thể
collection_name = "products_collection" # Thay bằng tên collection của bạn
collection = client.get_collection(name=collection_name)

# Lấy toàn bộ metadatas (chỉ include 'metadatas' để tối ưu RAM, không lấy vector embeddings)
results = collection.get(include=["metadatas"])
metadatas = results.get("metadatas", [])

# Lọc ra tập hợp các product_id độc nhất
unique_product_ids = set()
for meta in metadatas:
    if meta:
        # Lấy product_id hoặc id tuỳ thuộc cách bạn định nghĩa metadata lúc add vector
        p_id = meta.get("id") or meta.get("product_id")
        if p_id:
            unique_product_ids.add(p_id)

print(f"Thông tin thống kê của collection '{collection_name}':")
print(f"  -> Tổng số vector (chunk): {collection.count()}")
print(f"  -> Tổng số sản phẩm gốc độc nhất: {len(unique_product_ids)}")


Thông tin thống kê của collection 'products_collection':
  -> Tổng số vector (chunk): 1325
  -> Tổng số sản phẩm gốc độc nhất: 1325


In [22]:
# Lấy thử 5 vector đầu tiên
sample_data = collection.peek(limit=5)

for idx in range(len(sample_data['ids'])):
    print(f"\nVector ID: {sample_data['ids'][idx]}")
    print(f"  - Document: {sample_data['documents'][idx][:150]}...") # In 150 ký tự đầu
    print(f"  - Metadata: {sample_data['metadatas'][idx]}")



Vector ID: prod_136045
  - Document: Sản phẩm: Nubia Neo 5 GT Special Edition 12GB 256GB

Thương hiệu: Nubia | Danh mục: phone

Thông tin giá & Kho hàng:
- Giá thực tế: 13,990,000 VNĐ
- G...
  - Metadata: {'brand': 'Nubia', 'category': 'phone', 'price': 13990000.0, 'product_id': '136045'}

Vector ID: prod_135935
  - Document: Sản phẩm: Laptop Lenovo Yoga Slim 7 14IPH11 83QM0076VN

Thương hiệu: Lenovo | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 40,990,000 VN...
  - Metadata: {'price': 40990000.0, 'brand': 'Lenovo', 'product_id': '135935', 'category': 'laptop'}

Vector ID: prod_135930
  - Document: Sản phẩm: MacBook Pro M5 Pro 14 inch 2026 15CPU 16GPU 48GB 1TB Sạc 96W | Chính hãng Apple Việt Nam

Thương hiệu: Apple | Danh mục: laptop

Thông tin g...
  - Metadata: {'price': 71490000.0, 'product_id': '135930', 'category': 'laptop', 'brand': 'Apple'}

Vector ID: prod_135927
  - Document: Sản phẩm: MacBook Pro M5 Pro 14 inch 2026 15CPU 16GPU 48GB 1TB | Chính hãng Apple Việt